# Solar filament segmentation: reproduce the final submission

Two-stage pipeline: a **YOLO11m** detector proposes one box per filament, then a **box-conditioned ResNet-34 U-Net** segments each filament from a full-resolution crop. Details: `report/latex/main.tex`.

By default this notebook **does not train**. It loads the final weights stored in the repository (`weights/`) and writes the test submission:

1. Get the code and weights, and install the requirements
2. Locate the competition data
3. *(optional, `RETRAIN = True`)* retrain both models on all training images (about 3.5 h on one A10 GPU)
4. Predict the test set and write `submission.csv` (about 1 min on one GPU)
5. Check and visualise the predictions

Runs on Kaggle (attach the competition data) or any machine with the data and, ideally, a CUDA GPU.

In [ ]:
RETRAIN = False   # True: retrain both models from scratch instead of using weights/
DATA_ROOT = None  # folder holding the competition data; None = $FILSEG_DATA_ROOT or search /kaggle/input
REPO_URL = "github.com/blueclowd/solar_filaments_segmentation.git"
OUT = "submission.csv"

## 1. Code, weights and requirements
Uses the repository this notebook lives in; otherwise clones it (the weights are plain files in the repo, so a normal clone includes them).

In [ ]:
import os, subprocess, sys
from pathlib import Path

def find_repo():
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "weights" / "instance_r34.pt").exists() and (p / "filseg").is_dir():
            return p

REPO = find_repo()
if REPO is None:
    REPO = Path("/kaggle/working/solar_filaments_segmentation" if Path("/kaggle").exists()
                else "solar_filaments_segmentation").resolve()
    if not REPO.exists():
        token = os.environ.get("GITHUB_TOKEN", "")
        try:  # Kaggle secret, only needed while the repository is private
            from kaggle_secrets import UserSecretsClient
            token = token or UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            pass
        url = f"https://x-access-token:{token}@{REPO_URL}" if token else f"https://{REPO_URL}"
        r = subprocess.run(["git", "clone", "--depth", "1", url, str(REPO)], capture_output=True, text=True,
                           env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
        if r.returncode:
            raise RuntimeError("git clone failed: " + (r.stderr.replace(token, "***") if token else r.stderr))
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("repository:", REPO)
!ls -la weights

In [ ]:
!pip install -q -r requirements.txt
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only (slow)")

## 2. Competition data

In [ ]:
from filseg.config import resolve_data_root

if DATA_ROOT:
    os.environ["FILSEG_DATA_ROOT"] = str(DATA_ROOT)
root = resolve_data_root(os.environ.get("FILSEG_DATA_ROOT") or "/kaggle/input")
os.environ["FILSEG_DATA_ROOT"] = str(root)  # used by every script below
n_train = len(list((root / "train" / "train_images").iterdir()))
n_test = len(list((root / "test" / "test_images").iterdir()))
print(f"data: {root}\ntrain images: {n_train}   test images: {n_test}")

## 3. (Optional) retrain both models on all training images
Exactly the commands that produced `weights/`. Skipped unless `RETRAIN = True`. Post-processing uses the settings tuned on the validation split, stored in `configs/final_settings.json`.

- **Detector:** YOLO11m, 1536 px, 150 epochs, one label file per annotator, last epoch kept.
- **Segmenter:** ResNet-34 U-Net on 256x256 crops (image + box mask), BCE + Dice, 20 epochs.

Model selection and tuning (validation split, held-out months) are in `notebooks/kaggle_train.ipynb`.

In [ ]:
YOLO_WEIGHTS, SEG_WEIGHTS = "weights/yolo11m_1536.pt", "weights/instance_r34.pt"

if RETRAIN:
    ngpu = max(torch.cuda.device_count(), 1)
    devices = ",".join(map(str, range(ngpu)))
    !python train_yolo.py --full --model yolo11m.pt --imgsz 1536 --epochs 150 --batch {4 * ngpu} \
        --out-dir runs/yolo_full scale=0.25 device={devices}
    !python train_instance.py --full --config configs/instance_r34.yaml --output-dir runs/instance_full \
        --encoder resnet34 --epochs 20 --batch-size 32 --crop-size 256
    YOLO_WEIGHTS, SEG_WEIGHTS = "runs/yolo_full/train/weights/last.pt", "runs/instance_full/best.pt"
print("detector:", YOLO_WEIGHTS, "\nsegmenter:", SEG_WEIGHTS)

## 4. Predict the test set
Inference uses only the test images. Flip test-time augmentation is applied to each filament crop.

In [ ]:
!python predict_instances.py --split test --seg-run weights --seg-checkpoint {SEG_WEIGHTS} \
    --yolo-weights {YOLO_WEIGHTS} --settings configs/final_settings.json --out {OUT}

## 5. Check and visualise
Every RLE must decode to a 2048x2048 mask. Each colour is one predicted filament.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pycocotools import mask as mask_utils

sub = pd.read_csv(OUT)
sub["image"] = sub.filament_id.str.rsplit("_", n=1).str[0]
masks = [mask_utils.decode({"size": [2048, 2048], "counts": c.encode()}) for c in sub.segmentation_rle]
sub["area"] = [int(m.sum()) for m in masks]
per_image = sub.groupby("image").size()
print(f"{len(sub)} filaments in {sub.image.nunique()} of {n_test} test images")
print(f"filaments per image: median {per_image.median():.0f}, max {per_image.max()}")
print(f"mask area (px): min {sub.area.min()}, median {sub.area.median():.0f}, max {sub.area.max()}")
assert all(m.shape == (2048, 2048) for m in masks) and sub.area.min() > 0

In [ ]:
def overlay(stem, size=1024):
    path = next((root / "test" / "test_images").glob(stem + ".*"))
    img = cv2.cvtColor(cv2.imread(str(path), cv2.IMREAD_GRAYSCALE), cv2.COLOR_GRAY2RGB)
    rng = np.random.default_rng(0)
    for i in np.flatnonzero(sub.image.values == stem):
        color = tuple(int(c) for c in rng.integers(80, 256, 3))
        contours, _ = cv2.findContours(masks[i], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(img, contours, -1, color, 4)
    return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)

stems = per_image.sort_values().index[np.linspace(0, len(per_image) - 1, 4).astype(int)]  # few -> many filaments
fig, axes = plt.subplots(1, 4, figsize=(24, 6.5))
for ax, stem in zip(axes, stems):
    ax.imshow(overlay(stem)); ax.set_title(f"{stem}: {per_image[stem]} filaments"); ax.axis("off")
plt.tight_layout(); plt.show()

**Reproducibility note.** Inference runs in fp16 on GPU, so a different GPU model or library version can change a few boundary pixels; the filament list and scores should match the submitted file closely, not necessarily bit for bit.